# Варіант 1: Товар «Морозиво»
Практичне заняття: Візуалізація даних у Python (Matplotlib), аналіз розподілів та принципи чесної візуалізації

### Завдання 1. Побудова власного набору даних

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# 1. Фіксація генератора випадкових чисел за номером варіанта
variant = 1
np.random.seed(variant)

# 2. Генерація 30 днів місяця
days = np.arange(1, 31)

# 3. Визначення вихідних днів (кожен 6-й і 7-й день)
weekend = np.isin(days % 7, [6, 0])

# 4. Генерація температури (базова 22°C ± 3°C)
temperature = np.round(22 + np.random.normal(0, 3, size=30), 1)

# 5. Генерація продажів (базові 150 шт. + вплив температури + +20% у вихідні + випадкова варіація)
base_sales = 150
temp_effect = (temperature - 22) * 5  # вища температура збільшує продажі морозива
weekend_boost = np.where(weekend, base_sales * 0.20, 0) # +20% у вихідні
noise = np.random.normal(0, 8, size=30)

sales = np.round(base_sales + temp_effect + weekend_boost + noise).astype(int)

# 6. Створення DataFrame
df = pd.DataFrame({
    "день": days,
    "вихідний": weekend,
    "температура": temperature,
    "продажі": sales
})

print("--- Перші 5 рядків згенерованого датасету ---")
display(df.head())

### Завдання 2. Гістограма та вибір кількості bins

In [ ]:
# Побудова двох гістограм із різною кількістю bins
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Варіант A: bins = 6
ax1.hist(df["продажі"], bins=6, edgecolor="white", color="#2b5c8f")
ax1.set_xlabel("Обсяг продажів, шт.")
ax1.set_ylabel("Кількість днів")
ax1.set_title("Розподіл продажів (bins = 6)")

# Варіант B: bins = 15
ax2.hist(df["продажі"], bins=15, edgecolor="white", color="#d95f02")
ax2.set_xlabel("Обсяг продажів, шт.")
ax2.set_ylabel("Кількість днів")
ax2.set_title("Розподіл продажів (bins = 15)")

plt.tight_layout()
plt.show()

**Аналіз вибору кількості інтервалів (bins):**
- Для вибірки з $n = 30$ спостережень більш змістовну картину розподілу надає **`bins = 6`**.
- При `bins = 6` дані згруповані достатньо узагальнено, щоб чітко розпізнати форму розподілу (моду, центральну тенденцію) без випадкових прогалин.
- При `bins = 15` невеликий обсяг даних ($n = 30$) надмірно фрагментується: з'являються порожні або поодинокі інтервали, через що графік стає «зашумленим» і створює хибне враження про наявність кількох локальних піків у розподілі.

### Завдання 3. Boxplot: порівняння будніх і вихідних днів

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

# Формування двох груп
group_weekday = df.loc[~df["вихідний"], "продажі"]
group_weekend = df.loc[df["вихідний"], "продажі"]

ax.boxplot([group_weekday, group_weekend], labels=["Будні дні", "Вихідні дні"])
ax.set_ylabel("Обсяг продажів, шт.")
ax.set_title("Порівняння продажів у будні та вихідні дні")

plt.show()

print(f"Медіана продажів у будні: {group_weekday.median():.1f} шт.")
print(f"Медіана продажів у вихідні: {group_weekend.median():.1f} шт.")

**Аналіз boxplot:**
1. **Різниця медіан:** Медіана продажів у вихідні дні помітно вища, ніж у будні (приблизно 180–185 шт. проти 150–155 шт.). Це повністю узгоджується із заданою умовами Завдання 1 надбавкою на вихідні (+20%).
2. **Розкид (IQR):** Розкид центральних 50% даних (висота «ящика») в обох групах приблизно однаковий, що підтверджує рівномірність випадкової варіативності попиту незалежно від дня тижня.

### Завдання 4. Діаграма розсіювання (Sales vs Temperature)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.scatter(df["температура"], df["продажі"], color="#1b9e77", alpha=0.8)
ax.set_xlabel("Температура повітря, °C")
ax.set_ylabel("Обсяг продажів, шт.")
ax.set_title("Залежність продажів морозива від температури")

plt.show()

**Аналіз діаграми розсіювання:**
На графіку спостерігається чітко виражений **прямий (позитивний) лінійний зв'язок**: зі зростанням температури повітря обсяг продажів зростає. 
Це є цілком очікуваним результатом для товару «Морозиво», оскільки попит на охолоджувальні продукти безпосередньо залежить від спекотної погоди.

### Завдання 5. Навмисно спотворений графік (Порівняння з коректним)

In [ ]:
# Побудова порівняльної діаграми для середніх продажів (Коректна vs Спотворена)
mean_sales = df.groupby("вихідний")["продажі"].mean()

fig, (ax_honest, ax_distorted) = plt.subplots(1, 2, figsize=(12, 4))

# 1. Справний графік (з нуля)
ax_honest.bar(["Будні", "Вихідні"], mean_sales, color=["#4c72b0", "#55a868"])
ax_honest.set_ylabel("Середні продажі, шт.")
ax_honest.set_title("Коректна версія (вісь Y починається з 0)")
ax_honest.set_ylim(0, 210)

# 2. Спотворений графік (обрізана вісь Y)
ax_distorted.bar(["Будні", "Вихідні"], mean_sales, color=["#4c72b0", "#55a868"])
ax_distorted.set_ylabel("Середні продажі, шт.")
ax_distorted.set_title("Спотворена версія (обрізана вісь Y від 140)")
ax_distorted.set_ylim(140, 195)  # Обрізка нульової позначки!

plt.tight_layout()
plt.show()

**Аналіз застосованого спотворення (Лекція 8, Частина 1):**
У спотвореній версії було застосовано **усічення (обрізання) осі Y** (відсутність нульового базису / `zero baseline truncation`). 
- **Вплив на сприйняття:** Оскільки людський мозок порівнює довжину та площу стовпчиків, на спотвореному графіку стовпчик «Вихідні» здається приблизно у 3-4 рази вищим за стовпчик «Будні», хоча реальна різниця становить лише близько 20%. Коректний графік ліворуч демонструє дійсне масштабне співвідношення між групами.

### Відповіді на контрольні питання:

**1. Чи можете пояснити, чому обрізана вісь Y на стовпчиковій діаграмі перебільшує різницю між категоріями, хоча підписані числа лишаються правильними?**
Стовпчикова діаграма кодує числові значення через **довжину (висоту) та площу геометрії стовпчика**. Очі зорово порівнюють відношення висот $H_{вихідні} / H_{будні}$. Якщо вісь Y починається з 0, геометричне відношення висот відповідає дійсному відношенню чисел (наприклад, $180 / 150 = 1.2$). Якщо ж вісь Y обрізати й розпочати з 140, то висоти стовпчиків стануть $180 - 140 = 40$ та $150 - 140 = 10$, а їх зорове відношення складе $40 / 10 = 4.0$. Таким чином, графічне зображення створює ілюзію чотириразового переваження, незважаючи на правильно підписані текстові числа.

**2. Чи можете навести приклад, коли лінійний графік (з'єднання точок лінією) буде методологічною помилкою для категоріальних даних, и пояснити чому — з посиланням на шкали вимірювання з Лекції 4?**
Приклад: побудова лінійного графіка залежності середніх продажів від категорій міст («Київ», «Львів», «Одеса»).
*Обґрунтування:* Назви міст відносяться до **номінальної шкали вимірювання** (де елементи не мають природного порядку чи безперервності). Лінія між точками візуально імплікує наявність неперервної функції $y = f(x)$ та існування проміжних значень між категоріями. Оскільки проміжних значень між містами не існує, з'єднувальна лінія є методологічною помилкою, що викривляє природу даних.

**3. Чи розумієте, чому підписи осей, одиниці вимірювання і заголовок — не оздоблення графіка, а частина твердження, яке він робить?**
Графік без підписів осей, одиниць та заголовка є лише абстрактною геометричною фігурою. Підписи осей задають **контекст та просторові координати**, одиниці вимірювання визначають **фізичний або економічний масштаб** даних (наприклад, тисячі грн чи одиниці товару), а заголовок формулює **головну тезу/запитання** спостереження. Без цих елементів візуалізація втрачає семантичну цінність і не може виступати статистичним доказом.

**4. Чи можете пояснити, що таке chartjunk (наприклад, 3D-ефект на звичайній стовпчиковій діаграмі) і чому він шкодить, а не прикрашає сприйняття графіка?**
**Chartjunk («візуальне сміття»)** — це концепція, сформульована Едвардом Тафті, яка описує будь-які декоративні елементи графіка, що не несуть корисної інформації про дані (3D-об'єм, псевдоперспектива, фонові візерунки, важкі сітки, зайві яскраві кольори).
*Чому шкодить:* 3D-ефект спотворює кути зору та гранні площини стовпчиків, через що глядач не може точно визначити, по якій грані (передній чи задній) читати значення на осі Y. Зайві декорації знижують відношення «корисні чорнила / дані» (*data-ink ratio*) та відволікають від аналізу суті даних.